In [ ]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath('..'))

from quinex_application.visualisation.quinex_visualisation import QuinexVisualisation
from quinex_application.postprocessing.data_processing import reference_contains_term

In [ ]:
visualisation = QuinexVisualisation()

### **Import Results**

#### **CAPEX**

In [ ]:
cost_capex = pd.read_excel("../data/visualisation_files/unit_converted/df_technologies_capex.xlsx")
cost_capex["entity_mapping"].value_counts().head(20)

#### **Lifetime**

In [ ]:
df = pd.read_parquet("../data/visualisation_files/df_technologies.parquet")
lifetime = df[df["property_mapping"] == "Lifetime"]
lifetime = lifetime[lifetime['unit_normalized'].apply(
    lambda lst: len(lst) == 1 and lst[0].get('unit_uri') == 'http://qudt.org/vocab/unit/YR'
)]
lifetime["unit_normalized"].value_counts()

In [ ]:
# lifetime = lifetime[
#     lifetime['quantity_normalized'].apply(
#         lambda q: (
#             q.get('type') in ('single_quantity', 'range')
#             and all(qty.get('modifier') in ('=', '~') for qty in q.get('quantities', []))
#         )
#     )
# ]

#### **Efficiency**

In [ ]:
df = pd.read_parquet("../data/visualisation_files/df_technologies.parquet")
efficiency = df[df["property_mapping"] == "Efficiency"]
efficiency = efficiency[efficiency['unit_normalized'].apply(
    lambda lst: len(lst) == 1 and lst[0].get('unit_uri') == 'http://qudt.org/vocab/unit/PERCENT'
)]
efficiency["unit_normalized"].value_counts()

In [ ]:
# efficiency = efficiency[
#     efficiency['quantity_normalized'].apply(
#         lambda q: (
#             q.get('type') in ('single_quantity', 'range')
#             and all(qty.get('modifier') in ('=', '~') for qty in q.get('quantities', []))
#         )
#     )
# ]

#### **Capcity Factor**

In [ ]:
df = pd.read_parquet("../data/visualisation_files/df_technologies.parquet")
cap_factor_raw = df[df["property_mapping"] == "Capacity Factor"]
percent = cap_factor_raw[
    cap_factor_raw['unit_normalized'].apply(
        lambda lst: len(lst) == 1 and lst[0].get('unit_uri') == 'http://qudt.org/vocab/unit/PERCENT'
    )
]

unitless = cap_factor_raw[
    cap_factor_raw['unit_normalized'].apply(
        lambda lst: len(lst) == 1 and lst[0].get('unit_uri') == 'http://qudt.org/vocab/unit/UNITLESS'
    )
]

# nur Werte zwischen 0 und 1 behalten
unitless = unitless[unitless['value_normalized'].between(0, 1)]

# auf Prozent skalieren
unitless['value_normalized'] = unitless['value_normalized'] * 100

# zusammenführen
cap_factor = pd.concat([percent, unitless], ignore_index=True)
cap_factor["unit_normalized"].value_counts()

### **Visualize Results**

In [ ]:
cap_factor_dict_pv = {
    "year": list(range(2010, 2025)),
    "value": [15.0, 15.3, 15.1, 16.5, 16.6, 16.5, 16.8, 17.5, 17.9, 17.5, 16.1, 17.2, 16.9, 16.2, 17.4],
    "entity": ["PV"] * 15
}
cap_factor_dict_onshore = {
    "year": list(range(2010, 2025)),
    #"value": [29, 31, 33, 35, 36, 36, 39, 37, 36, 34],
    "value": [27, 27, 28, 28, 29, 29, 31, 33, 35, 36, 36, 39, 37, 36, 34],
    "entity": ["Onshore Wind"] * 15
}
cap_factor_dict_offshore = {
    "year": list(range(2015, 2025)),
    "value": [42, 40, 45, 39, 42, 39, 39, 41, 41, 42],
    # "value": [38, 38, 40, 44, 35, 42, 40, 45, 39, 42, 39, 39, 41, 41, 42],
    "entity": ["Offshore Wind"] * 10
}

cap_factor_additional = pd.concat([pd.DataFrame(cap_factor_dict_pv), pd.DataFrame(cap_factor_dict_onshore), pd.DataFrame(cap_factor_dict_offshore)], ignore_index=True)

In [ ]:
capex_dict_pv = {
    "year": list(range(2010, 2025)),
    "value": [5.284, 4.659, 3.537, 3.107, 2.849, 2.140, 1.958, 1.683, 1.438, 1.188, 1.044, 0.973, 0.930, 0.766, 0.691],
    "entity": ["PV"] * 15
}

capex_additional_onshore = {
    "year": list(range(2010, 2025)),
    "value": [2.324, 2.278, 2.165, 2.234, 2.147, 1.956, 1.960, 1.963, 1.865, 1.766, 1.589, 1.506, 1.353, 1.187, 1.041],
    "entity": ["Onshore Wind"] * 15
}

# capex_additional_offshore = {
#     "year": list(range(2010, 2025)),
#     "value": [5.518, 6.343, 5.651, 5.819, 6.247, 6.265, 4.933, 5.531, 4.944, 4.354, 3.87, 3.229, 3.518, 2.867, 2.852],
#     "entity": ["Offshore Wind"] * 15
# }
capex_additional_offshore = {
    "year": list(range(2019, 2025)),
    "value": [4.354, 3.87, 3.229, 3.518, 2.867, 2.852],
    "entity": ["Offshore Wind"] * 6
}

capex_additional = pd.concat([pd.DataFrame(capex_dict_pv), pd.DataFrame(capex_additional_onshore), pd.DataFrame(capex_additional_offshore)], ignore_index=True)

In [ ]:
lcoe_dict_pv = {
    "year": list(range(2010, 2025)),
    "value": [0.417, 0.350, 0.260, 0.200, 0.183, 0.135, 0.118, 0.093, 0.079, 0.069, 0.061, 0.055, 0.051, 0.042, 0.043],
    "entity": ["PV"] * 15
}

lcoe_additional_onshore = {
    "year": list(range(2010, 2025)),
    "value": [0.113, 0.107, 0.098, 0.098, 0.088, 0.076, 0.070, 0.066, 0.056, 0.049, 0.041, 0.036, 0.034, 0.033, 0.034],
    "entity": ["Onshore Wind"] * 15
}

lcoe_additional_offshore = {
    "year": list(range(2010, 2025)),
    "value": [0.208, 0.215, 0.185, 0.156, 0.190, 0.155, 0.128, 0.117, 0.114, 0.095, 0.091, 0.082, 0.081, 0.076, 0.079],
    "entity": ["Offshore Wind"] * 15
}

lcoe_additional = pd.concat([pd.DataFrame(lcoe_dict_pv), pd.DataFrame(lcoe_additional_onshore), pd.DataFrame(lcoe_additional_offshore)], ignore_index=True)

In [ ]:
cost_capex['entity_mapping'] = cost_capex['entity_mapping'].replace({
    'Wind': 'Onshore Wind'
})
cap_factor['entity_mapping'] = cap_factor['entity_mapping'].replace({
    'Wind': 'Onshore Wind'
})
print(len(cap_factor[cap_factor["entity_mapping"].isin((['Onshore Wind']))]))

df_sample = cap_factor[cap_factor['entity_mapping'] == 'Onshore Wind'].sample(n=500)
df_sample = df_sample.applymap(
    lambda x: f"'{x}" if isinstance(x, str) and x.startswith('=') else x
)
df_sample.to_csv("capfactor_data_500.csv", index=False, sep=';', decimal=',')


data = [
    {
    'df': cost_capex[cost_capex["entity_mapping"].isin(['PV', 'Offshore Wind', 'Onshore Wind'])],
    'entity_col': 'entity_mapping',
    'value_col': 'value_converted',
    'ylabel': 'CAPEX [$/kW]', 
    'title': 'a',
    'additional_data': capex_additional,
    'year_range': (2007, 2051),
    'y_range': (0, 7),
    },
    {
    'df': cap_factor[cap_factor["entity_mapping"].isin((['PV', 'Offshore Wind', 'Onshore Wind']))],
    'entity_col': 'entity_mapping',
    'value_col': 'value_normalized',
    'ylabel': 'Capacity Factor [%]', 
    'title': 'b',
    'additional_data': cap_factor_additional,
    'year_range': (2007, 2051),
    'y_range': (0, 60),
    },
    {
    'df': efficiency[efficiency["entity_mapping"].isin(['Lithium-Ion Battery', 'Lead Acid Battery'])],
    'entity_col': 'entity_mapping',
    'value_col': 'value_normalized',
    'ylabel': 'Efficiency [%]', 
    'title': 'c',
    'year_range': (2014, 2026),
    'y_range': (80, 95),
    },
    {
    'df': lifetime[lifetime["entity_mapping"].isin(['Lithium-Ion Battery', 'Lead Acid Battery'])],
    'entity_col': 'entity_mapping',
    'value_col': 'value_normalized',
    'ylabel': 'Lifetime [Years]',
    'title': 'd',
    'year_range': (2014, 2026),
    'y_range': (5, 15),
    },
    {
    'df': efficiency[efficiency["entity_base_mapping"].isin(['PV', 'Battery', 'Wind', 'Electrolyzer', 'Coal'])],
    'entity_col': 'entity_base_mapping',
    'value_col': 'value_normalized',
    'ylabel': 'Efficiency [%]', 
    'title': 'e',
    'year_range': (2007, 2051),
    'y_range': (0, 95),
    },
    {
    'df': lifetime[lifetime["entity_base_mapping"].isin(['PV', 'Battery', 'Wind', 'Electrolyzer', 'Coal'])],
    'entity_col': 'entity_base_mapping',
    'value_col': 'value_normalized',
    'ylabel': 'Lifetime [Years]',
    'title': 'f',
    'year_range': (2007, 2051),
    'y_range': (0, 85),
    },
]


fig, data_tables = visualisation.plot_multiple_properties_over_time(
    data_list = data,
    legend_title = 'Technology Type',
    color_dict=visualisation.entity_colors,
    figsize=(24,18),
    legend_position=(0.97,0.95),
    legend_size_position=(0.97,0.53),
    central_tendency="moving average",
    error_type="se",
    error_display="shade",
    font_size=30,
    point_size_range=(300, 900),
    min_papers_per_year=3,
    min_datapoints_per_year=3,
    )

fig.show()
#fig.savefig("../data/figures/results/temporal/technologies_properties.pdf", bbox_inches='tight')